# KeyDiff Token Dropping Example with LMCache SDK
---

Long prompts create large KV caches that eat up GPU memory and limit how many
requests fit in a batch, and a smaller batch means lower decode throughput.
Token dropping shrinks each request's KV cache (in this example by half), so
more requests fit in a batch.

This notebook uses **KeyDiff** ([Park et al., 2025](https://arxiv.org/abs/2504.15364))
to choose which tokens to keep. KeyDiff scores tokens from the **keys alone**:
a token whose key points in a different direction from the average key is
distinctive, so it is kept. Unlike the SnapKV and R-KV examples, it does not
read the query tensor, so scoring is cheaper. The SDK still moves the KV cache
through vLLM's query ring, so the setup is the same as the SnapKV example: the
vLLM patch from [README.md](../README.md), the qstore server module, and
`lmcache.mp.transfer_intermediate_tensors`. Like the SnapKV
example, it also measures answer accuracy before and after dropping.

On a Colab T4 with `Qwen/Qwen3-1.7B` and the short dataset (10 prompts), two
runs gave decode throughput of 127.2 -> 196.1 and 131.7 -> 193.1 tokens/s
(about 1.5x), and accuracy of 8/10 -> 7/10 in both. With 10 prompts, one answer
is a small difference.

The prefill step samples its single token greedily (`temperature`), like
`rkv_colab.ipynb`. That token becomes the first token of the answer, so sampling
it at temperature 1.0 would make accuracy change from run to run.

Requirements:
* The 10-line vLLM patch that exposes intermediate tensors to LMCache. Follow
  the setup steps in [README.md](../README.md).
* This experiment requires a GPU. To demonstrate how token dropping increases
  the decoding batch size, adjust the GPU memory utilization together with the
  number of requests.
* This example uses shared memory for data transfer between LMCache server and
  the SDK. If shared memory is unavailable, the SDK automatically falls back to pickle.


## Setup vLLM and LMCache server
Follow below instructions (1-4), then proceed to below Python cell.

**1. Start LMCache server first**

To use shared memory, specify the `--shm-name` and `--no-l1-use-lazy`.
Load the qstore server module with `--server-module`; the SDK needs it even
though KeyDiff does not read the query tensor.

```sh
lmcache server \
    --l1-size-gb 150 \
    --eviction-policy LRU \
    --chunk-size 256 \
    --port 6555 \
    --http-port 8080 \
    --shm-name lmcache_kvcache_sdk_e2e \
    --no-l1-use-lazy \
    --server-module '{"module_path": "lmcache.v1.multiprocess.plugin_modules.qstore"}' \
    --supported-transfer-mode auto
```

**2. Wait until LMCache server is ready**

```sh
curl -sf http://localhost:8080/healthcheck && echo " LMCache ready"
```

**3. Start vLLM once LMCache server is ready**

We pass --no-enable-prefix-caching to disable vLLM's built-in prefix caching. 
This ensures the prefilled KV cache is always served from LMCache rather than 
vLLM, so the decoding throughput improvement can be attributed entirely to the 
tokens dropped through LMCache.

```sh
vllm serve Qwen/Qwen3-8B \
    --port 8000 \
    --served-model-name Qwen/Qwen3-8B \
    --no-enable-prefix-caching \
    --enforce-eager \
    --gpu-memory-utilization 0.65 \
    --kv-transfer-config '{"kv_connector":"LMCacheMPConnector","kv_role":"kv_both","kv_connector_extra_config":{"lmcache.mp.port":6555,"lmcache.mp.transfer_intermediate_tensors":true}}' \
    --trust-remote-code \
    --return-tokens-as-token-ids
```

On a smaller GPU, use `Qwen/Qwen3-1.7B` here and in `model_name` below, and the
`raniayu/token-dropping-demo-short` dataset. On a Colab T4, follow the setup in
[rkv_colab.ipynb](../rkv_colab.ipynb) (vLLM 0.23.0, `--attention-backend TRITON_ATTN`,
`VLLM_USE_FLASHINFER_SAMPLER=0`, `max_tokens = 256`).

**4. Wait until vLLM is ready**

```sh
curl -sf http://localhost:8000/v1/models && echo " vLLM ready"
```

In [ ]:
# SPDX-License-Identifier: Apache-2.0
"""End-to-end KV cache remapping driver for the SDK example."""

# Standard
from datasets import load_dataset
import matplotlib.pyplot as plt
import sys
from transformers import AutoTokenizer, AutoConfig

# Third Party
import torch
import torch.nn.functional as F

# First Party
from lmcache.logging import init_logger
import lmcache.sdk as lmc_sdk
from lmcache.banner import print_banner_once

sys.path.append("..")  # utils.py lives next to the main notebooks

from utils import (  # noqa: E402
    rerotate_k_cache,
    make_post_completion,
    score_answers,
)

print_banner_once(sys.stdout)
logger = init_logger(__name__)

## Setting up hyperparameters

In [ ]:
model_name = "Qwen/Qwen3-8B"
vllm_url = "http://localhost:8000"
lmcache_url = "http://localhost:8080"
lmcache_mq_url = "tcp://localhost:6555"
chunk_size = 256
max_tokens = 2048
temperature = 0.0  # greedy decode so generated answers are deterministic and scorable
timeout = 60  # timeout for context retrieval (seconds)
trust_remote_code = True

## Configuring the model and LMCache endpoint

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    model_name, trust_remote_code=trust_remote_code
)
config = AutoConfig.from_pretrained(model_name, trust_remote_code=trust_remote_code)
head_size = getattr(
    config, "head_dim", config.hidden_size // config.num_attention_heads
)
work_device = torch.device("cpu")
post_completion = make_post_completion(vllm_url, model_name, timeout)
ctx = lmc_sdk.kvcache.connect(
    url=lmcache_mq_url,
    http_url=lmcache_url,
    model_name=model_name,
    timeout=timeout,
)

## Constructing Prompt

In [ ]:
prompts = []
answers = []
ds = load_dataset("raniayu/token-dropping-demo", split="train")
for i, example in enumerate(ds):
    prompt = tokenizer.encode(ds[i]["prompt"], return_tensors="pt").squeeze(0).tolist()
    prompts.append(prompt)
    answers.append(ds[i]["answer"])

print(f"Loaded {len(prompts)} prompts from the dataset.")
print(f"Total prompts length: {sum(len(p) for p in prompts)} tokens.")
print(f"Mean prompt length: {sum(len(p) for p in prompts) / len(prompts):.2f} tokens.")
print(f"Max prompt length: {max(len(p) for p in prompts)} tokens.")
print(f"Min prompt length: {min(len(p) for p in prompts)} tokens.")

## Baseline (without token dropping)

In [ ]:
batch = lmc_sdk.batch.LMCacheBatchedStream()
for i, prompt in enumerate(prompts):
    request = lmc_sdk.request.create_request(
        contexts=[ctx],
        post_completion=post_completion,
        prompt_token_ids=prompt,
    )
    batch.add(request)
results = batch.prefill(
    sampling_params={"max_tokens": 1, "temperature": temperature, "ignore_eos": True}
)
results.emit()

logger.info("\n\n")

results = batch.decode(
    sampling_params={
        "max_tokens": max_tokens,
        "temperature": temperature,
        "ignore_eos": True,
    }
)
results.emit()
before_drop_data = results.to_dict()
before_texts = [request.output_text for request in batch.request_streams.values()]

## Clear LMCache

In [ ]:
!curl -X POST {lmcache_url}/cache/clear

## With Token Dropping

For every layer and every KV head, each token's key is compared with the
head's average (normalized) key. The score is the negative cosine similarity:
the more a key differs from the average, the higher its score. Scores are
averaged over heads and layers, the same way the SnapKV example averages its
attention scores, so one set of tokens is kept for the whole request.

The first `keydiff_num_sinks` tokens (attention sinks) and the last
`keydiff_window_size` tokens (which hold the question) are always kept. From
the rest, the highest-scoring tokens are kept until half of the prompt remains.


In [ ]:
keydiff_drop_ratio = 0.5  # drop half of the tokens, like the other examples
keydiff_window_size = 64  # always keep the most recent tokens (the question)
keydiff_num_sinks = 4  # always keep the first tokens (attention sinks)


def _keydiff_layer_scores(
    k_flat: torch.Tensor, *, seq_len: int, num_kv_heads: int, head_size: int
) -> torch.Tensor:
    """KeyDiff score of every token in one layer, averaged over KV heads.

    A token whose key points in a different direction from the head's average
    key is distinctive and worth keeping, so the score is the negative cosine
    similarity to the mean of the normalized keys. Higher = keep.
    """
    k = k_flat[:seq_len].to(device=work_device, dtype=torch.float32)
    k = k.reshape(seq_len, num_kv_heads, head_size).permute(1, 0, 2)  # [H, T, d]
    anchor = F.normalize(k, p=2, dim=-1).mean(dim=1, keepdim=True)  # [H, 1, d]
    return -F.cosine_similarity(k, anchor, dim=-1).mean(dim=0)  # [T]


def drop_tokens_fn(
    tensors: dict[lmc_sdk.context.LMCacheSDKCacheKind, torch.Tensor],
    token_source: list[int],
) -> tuple[torch.Tensor, list[int]]:
    """KeyDiff token dropping: keep the most distinctive keys (KV cache only)."""
    kv_tensor = tensors[lmc_sdk.context.LMCacheSDKCacheKind.KV]
    seq_len = min(kv_tensor.shape[2], len(token_source))

    num_q_heads = config.num_attention_heads
    num_kv_heads = getattr(config, "num_key_value_heads", num_q_heads)
    head_dim = getattr(config, "head_dim", config.hidden_size // num_q_heads)
    if kv_tensor.shape[-1] != num_kv_heads * head_dim:
        raise ValueError(
            f"KV hidden dim {kv_tensor.shape[-1]} != {num_kv_heads * head_dim}"
        )

    middle_start = keydiff_num_sinks
    middle_end = seq_len - keydiff_window_size
    if middle_end <= middle_start:
        raise ValueError(
            f"seq_len ({seq_len}) must exceed sinks + window "
            f"({keydiff_num_sinks + keydiff_window_size})"
        )

    total_keep = int(round(seq_len * (1 - keydiff_drop_ratio)))
    keep_middle = max(
        0,
        min(
            total_keep - keydiff_num_sinks - keydiff_window_size,
            middle_end - middle_start,
        ),
    )

    # Average the per-layer scores, as the SnapKV example does with attention scores.
    scores = torch.zeros(seq_len, dtype=torch.float32, device=work_device)
    for layer in range(kv_tensor.shape[1]):
        scores += _keydiff_layer_scores(
            kv_tensor[0, layer],  # keys of this layer: [T, num_kv_heads * head_dim]
            seq_len=seq_len,
            num_kv_heads=num_kv_heads,
            head_size=head_dim,
        )
    scores /= kv_tensor.shape[1]

    middle_keep = (
        torch.topk(scores[middle_start:middle_end], k=keep_middle).indices
        + middle_start
    )
    keep_idx = torch.cat(
        [
            torch.arange(middle_start, device=work_device),
            middle_keep.sort().values,
            torch.arange(middle_end, seq_len, device=work_device),
        ]
    ).to(dtype=torch.long)

    keep_idx_cpu = keep_idx.cpu()
    kept_ids = [token_source[i] for i in keep_idx_cpu.tolist()]

    e_kv = rerotate_k_cache(
        kv_tensor[:, :, keep_idx_cpu, :].clone().to(work_device),
        old_positions=keep_idx,
        new_positions=torch.arange(keep_idx.numel(), device=work_device),
        model_config=config,
    )

    logger.info("compacted %s/%s tokens.", e_kv.shape[2], kv_tensor.shape[2])

    return e_kv.cpu(), kept_ids

In [ ]:
stream_ids = []

batch = lmc_sdk.batch.LMCacheBatchedStream()
for i, prompt in enumerate(prompts):
    request = lmc_sdk.request.create_request(
        contexts=[ctx],
        post_completion=post_completion,
        prompt_token_ids=prompt,
    )
    batch.add(request)
    stream_ids.append(request.request_stream_id)

results = batch.prefill(
    sampling_params={"max_tokens": 1, "temperature": temperature, "ignore_eos": True}
)
results.emit()

logger.info("\n\n")

results = batch.modify(drop_tokens_fn)
results.emit()

logger.info("\n\n")

results = batch.decode(
    sampling_params={
        "max_tokens": max_tokens,
        "temperature": temperature,
        "ignore_eos": True,
    }
)
results.emit()
after_drop_data = results.to_dict()
after_texts = [request.output_text for request in batch.request_streams.values()]

### Accuracy

In [ ]:
before_correct, total, before_map = score_answers(answers, before_texts)
after_correct, total, after_map = score_answers(answers, after_texts)
logger.info(
    "Baseline (no drop): %s/%s correct (%.1f%%)",
    before_correct,
    total,
    100 * before_correct / total,
)
logger.info(
    "Token dropping: %s/%s correct (%.1f%%)",
    after_correct,
    total,
    100 * after_correct / total,
)

## Result Plot

In [ ]:
labels = ["Without", "With"]
data = [
    before_drop_data["metrics"]["results"]["output_tput"],
    after_drop_data["metrics"]["results"]["output_tput"],
]

fig, ax = plt.subplots(figsize=(2.5, 3))
bars = ax.bar(labels, data, color=["blue", "orange"])
ax.bar_label(bars, fmt="%.1f", padding=3)
ax.set_ylabel("Decode Throughput (tokens/s)")
ax.set_title("Token Dropping Benefit\nfor Decode Throughput")
ax.margins(y=0.15)
plt.tight_layout()
plt.show()

## Close the context

In [ ]:
# Only close the context when done.
ctx.close()